# Base general: Informacion de ingresos y egresos hospitalarios 
En este notebook se visualizan los datos de ingresos y egresos hospitalarios de Salud
Se utilizan datos de 2015 a 2024, y se visualizan los flujos de pacientes entre provincias, asi como las causas mas comunes de ingreso hospitalario.



In [ ]:
import polars as pl
import gc

In [ ]:
base_general=pl.read_csv("../../data/data_hospitales/sistema_salud_egresos.csv",infer_schema=0)

In [ ]:
base_general.schema

In [ ]:
base_general.describe()

In [ ]:
identificadores = [
    'clase',
    'tipo',
    'entidad',
    'sector'
]
territorio_ubi=[    
    'prov_ubi',
    'cant_ubi',
    'parr_ubi',
    #'area_ubi',

]
territorio_res = [
    'prov_res',
    'cant_res',
    'parr_res',
    #'area_res',
]

fecha_egr = [
    'fecha_egr',

]

estadisticas = [
    'dia_estad', #dias de estadia
    'con_egrpa', # egreso vivo o muerto (antes o despues de 48 horas)
]

diagnostico=[
    "cau_cie10",
    "causa3",
    "cap221rx",
    "cau221rx",
    "cau298rx",
]

demografia=[
    "sector",
    "mes_inv",
    "nac_pac",
    "nom_pais",
    "cod_pais",
    "sexo",
    "cod_edad",
    "edad",
    "etnia",
    
]




In [ ]:
base_general["cod_edad"].value_counts(sort=True)

In [ ]:
# Obtener los value_counts de 'col1' agrupados por 'col2'
result = base_general.group_by("cod_edad").agg([
    pl.col("edad").value_counts().alias("edad_counts")
])

print(result)

In [ ]:
# Función para mapear las edades
def map_edad_to_years(cod_edad, edad):
    if cod_edad == "Horas (1 a 23 horas de edad)":
        return float(edad) / 24 / 365  # Convertir horas a años
    elif cod_edad == "Días (1 a 28 días de edad)":
        return float(edad) / 365  # Convertir días a años
    elif cod_edad == "Meses (1 a 11 meses de edad)":
        return float(edad) / 12  # Convertir meses a años
    elif cod_edad == "Años":
        return float(edad)  # Ya está en años
    return float(edad)

# Crear una nueva columna con la edad estandarizada

# # Función de mapeo

# def map_if_has_code(cau_value,cie_code):
#     return mapping_with_code.get((cau_value, cie_code), cie_code)

# Aplicar el mapeo utilizando map_batches
base_general = base_general.with_columns(
    pl.struct(["cod_edad", "edad"])  # Crear un struct de las dos columnas
    .map_elements(  # Aplicar la función en batches
        lambda batch: map_edad_to_years(
            batch.get("cod_edad"),
            batch.get("edad"),
            ),  # Aplicar la función
        return_dtype=pl.Float64,  # El tipo de dato de retorno es cadena
    ).alias("edad_std")  # Asignar la nueva columna
)


In [ ]:
columns_to_extract = (identificadores+territorio_ubi + territorio_res + fecha_egr + estadisticas + diagnostico+["edad_std"])
#base_general_filtrada=base_general.select(columns_to_extract)
base_general_filtrada=base_general.clone()
del base_general
gc.collect()

In [ ]:

edades=base_general_filtrada["edad_std"].value_counts(sort=True)

#### Mapeo cau221rx

In [ ]:
# leer diccionario del archivo: notebooks/limpieza/nuevo_diccionario_cau221rx.txt
import ast

# 1. Leemos el contenido del archivo .txt
with open("nuevo_diccionario_cau221rx.txt", "r", encoding="utf-8") as f:
    contenido = f.read()

# 2. Limpiamos el texto para quedarnos solo con lo que está después del '='
# Esto elimina "dictionary = " y nos deja solo con el "{...}"
if "=" in contenido:
    str_diccionario = contenido.split("=", 1)[1].strip()
else:
    str_diccionario = contenido.strip()

# 3. Lo convertimos en un objeto diccionario real
nuevo_diccionario = ast.literal_eval(str_diccionario)

# Ahora ya puedes usarlo normalmente
print(type(nuevo_diccionario))
print(nuevo_diccionario["129 Artropatías infecciosas (M00-M03)"])

In [ ]:

base_general_filtrada=base_general_filtrada.with_columns(
    pl.col("cau221rx")
    .replace(nuevo_diccionario, default=None)
    .alias("cau221rx_std")
)

In [ ]:
base_general_filtrada["cau221rx_std"].value_counts()

In [ ]:
base_general_filtrada.select(pl.col("cau221rx_std").is_null().sum())


In [ ]:
# see wich cau221rx doesnt have a match in cau221rx_dict
cau221rx_not_maped=base_general_filtrada.filter(base_general_filtrada["cau221rx_std"].is_null())

In [ ]:
cau221rx_review = (
    base_general_filtrada
    .group_by("cau221rx") # Agrupamos por el código
    .len()                     # Contamos (en versiones viejas usa .count())
    .sort("len", descending=True) # Ordenamos de mayor a menor
)

display(cau221rx_review)


## Mapeo cie10

##### Intento en polars

In [ ]:
# import re
# import pandas as pd
# from rapidfuzz import fuzz, process


# def build_canonical_mapping(values, similarity_threshold=90):
#     uniques = sorted(set(values))
#     mapping = {}   # old -> canonical
#     seen = set()   # already assigned to some cluster

#     for val in uniques:
#         if val in seen:
#             continue

#         # Find similar strings
#         matches = process.extract(
#             val,
#             uniques,
#             scorer=fuzz.token_sort_ratio,
#             score_cutoff=similarity_threshold
#         )

#         group_vals = {v for v, score, idx in matches}
#         group_vals.add(val)

#         # choose the "most informative" one — here, longest string
#         canonical_val = max(group_vals, key=len)

#         for v in group_vals:
#             mapping[v] = canonical_val
#             seen.add(v)

#     return mapping

# # Extracción de código CIE
# def extract_cie_code(text):
#     if text is None:
#         return None
#     text = str(text)
#     m = re.search(r'\b([A-Z]\d{2,3}[A-Z0-9]?)\b', text)
#     return m.group(1) if m else ""

    
# def map_no_code_row(row):
#     if row["cie_code"] is not None:
#         return row["cau_cie10_temp"]
    
#     cau = row["cau221rx_std"]
#     text = str(row["cau_cie10"])
    
#     if cau not in candidates_by_cau:
#         return row["cau_cie10_temp"]
    
#     candidates = list(candidates_by_cau[cau])
    
#     # Buscar el candidato más similar
#     match, score, _ = process.extractOne(
#         text, candidates, scorer=fuzz.token_sort_ratio
#     )
    
#     if score >= SIM_THRESHOLD:
#         return match
#     else:
#         return row["cau_cie10_temp"]

# # DataFrame original en Polars
# base_genera_cie = base_general_filtrada[["cau221rx_std", "cau_cie10"]].unique(maintain_order=True)

# # Agregar el código CIE
# base_genera_cie = base_genera_cie.with_columns(
#     pl.col("cau_cie10").map_elements(extract_cie_code,return_dtype=pl.Utf8).alias("cie_code")
# )
# # Crear el mapeo con códigos
# mapping_with_code = {}

# # Filtrar las filas que contienen códigos
# df_with_code = base_genera_cie.filter(pl.col("cie_code").is_not_null())
# df_no_grouped=df_with_code.clone()

# # Cambia esta línea:
# # for cau_value, cie_code in df_with_code.group_by(["cau221rx_std", "cie_code"]):

# # Por esta:
# for group in df_with_code.group_by(["cau221rx_std", "cie_code"]):
#     # Desempaquetar el grupo
#     (cau_value, cie_code), subdf = group
    
#     # Obtener los valores únicos de cau_cie10
#     values = subdf["cau_cie10"].unique()
    
#     if len(values) == 0:
#         continue
    
#     # Regla: el texto más largo es el más informativo
#     canonical = max(values, key=len)
    
#     for v in values:
#         mapping_with_code[(cau_value, v)] = canonical



In [ ]:


# # Función de mapeo

# def map_if_has_code(cau_value,cie_code):
#     return mapping_with_code.get((cau_value, cie_code), cie_code)

# # Aplicar el mapeo utilizando map_batches
# base_genera_cie = base_genera_cie.with_columns(
#     pl.struct(["cau221rx_std", "cau_cie10"])  # Crear un struct de las dos columnas
#     .map_elements(  # Aplicar la función en batches
#         lambda batch: map_if_has_code(
#             batch.get("cau221rx_std"),
#             batch.get("cau_cie10"),
#             ),  # Aplicar la función
#         return_dtype=pl.Utf8,  # El tipo de dato de retorno es cadena
#     ).alias("cau_cie10_temp")  # Asignar la nueva columna
# )

# # Asegúrate de usar map_elements y NO map_batches
# base_genera_cie = base_genera_cie.with_columns(
#     pl.struct(["cau221rx_std", "cau_cie10"])
#     .map_elements(
#         lambda x: mapping_with_code.get((x["cau221rx_std"], x["cau_cie10"]), x["cau_cie10"]),
#         return_dtype=pl.String
#     )
#     .alias("cau_cie10_temp")
# )

# # Función de mapeo
# def map_if_has_code(struct):
#     cau_value = struct.field("cau221rx_std")  # Extraer el valor de cau221rx_std
#     cie_code = struct.field("cau_cie10")        # Extraer el valor de cau_cie10
#     return mapping_with_code.get((cau_value, cie_code), cie_code)

# # Aplicar el mapeo utilizando map_batches
# base_genera_cie = base_genera_cie.with_columns(
#     pl.struct(["cau221rx_std", "cau_cie10"])  # Crear un struct de las dos columnas
#     .map_batches(  # Aplicar la función en batches
#         lambda batch: batch.apply(lambda struct: map_if_has_code(struct)),  # Aplicar la función
#         return_dtype=pl.Utf8  # El tipo de dato de retorno es cadena
#     ).alias("cau_cie10_temp")  # Asignar la nueva columna
# )

# SIM_THRESHOLD = 90



# # Aplicar el mapeo a filas sin código
# base_genera_cie = base_genera_cie.with_columns(
#     pl.col("cau_cie10").apply(map_no_code_row).alias("cau_cie10_std")
# )

        
# # Crear el mapeo final de cau_cie10 limpio
# mapping_ciie10 = {}
# for cie_std, cie, subdf in base_genera_cie.groupby(["cau_cie10_std", "cau_cie10"]):
#     mapping_ciie10[cie] = cie_std

# # Aplicar el mapeo final a base_general_filtrada
# base_general_filtrada = base_general_filtrada.with_columns(
#     pl.col("cau_cie10").map(mapping_ciie10).alias("cau_cie10_std")
# )



##### Implementacion en Pandas


In [ ]:
base_general_filtrada = base_general_filtrada.with_columns(
    pl.col("cau_cie10")
    .str.replace_all('A"', "O")  # Caso: A"RGANO -> ORGANO
    .str.replace_all('"', "O")   # Caso: SJ"GREN -> SJOGREN / WALDENSTR"M -> WALDENSTROM        # Limpieza final de espacios
)

In [ ]:
base_general_filtrada_pd=base_general_filtrada.to_pandas()

In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process
import pandas as pd
import re

def build_canonical_mapping(values, similarity_threshold=90):
    uniques = sorted(set(values))
    mapping = {}   # old -> canonical
    seen = set()   # already assigned to some cluster

    for val in uniques:
        if val in seen:
            continue

        # Find similar strings
        matches = process.extract(
            val,
            uniques,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=similarity_threshold
        )

        group_vals = {v for v, score, idx in matches}
        group_vals.add(val)

        # choose the "most informative" one — here, longest string
        canonical_val = max(group_vals, key=len)

        for v in group_vals:
            mapping[v] = canonical_val
            seen.add(v)

    return mapping

# --- 1. Configuración y Funciones Base ---
SIM_THRESHOLD = 90  # Ajusta según necesites

def extract_cie_code(text):
    if pd.isna(text):
        return None
    text = str(text)
    m = re.search(r'\b([A-Z]\d{2,3}[A-Z0-9]?)\b', text)
    return m.group(1) if m else None

def get_clean_text(text):
    """Elimina el código CIE del texto para una comparación limpia"""
    if pd.isna(text): return ""
    text = str(text)
    code = extract_cie_code(text)
    if code:
        # Reemplaza el código por nada y limpia espacios sobrantes
        return re.sub(rf'\b{re.escape(code)}\b', '', text).strip()
    return text.strip()



In [ ]:
import ast
from pathlib import Path

# 1. Definimos la ruta usando Path para evitar líos con las diagonales
ruta_archivo = Path("mapping_cie10.txt")

# 2. Verificamos si el archivo existe
if ruta_archivo.exists():
    with open(ruta_archivo, "r", encoding="utf-8") as f:
        contenido = f.read()
    
    # 3. Extraemos solo la parte del diccionario (después del '=')
    if "=" in contenido:
        str_diccionario = contenido.split("=", 1)[1].strip()
    else:
        str_diccionario = contenido.strip()
    
    # 4. Intentamos convertirlo a objeto real de Python
    try:
        mapping_final_dict = ast.literal_eval(str_diccionario)
        print(f"✅ Diccionario cargado con {len(mapping_final_dict)} entradas.")

        # Ahora ya puedes usarlo normalmente
        print(type(mapping_final_dict))
        print(mapping_final_dict["T529 EFECTO TOXICO DE DISOLVENTES ORGANICOS, NO ESPECIFICADOS"])
    except (ValueError, SyntaxError) as e:
        print(f"❌ Error: El contenido del archivo no es un diccionario válido. {e}")
        mapping_final_dict = {}
else:
    
    
    print(f"⚠️ El archivo no existe en la ruta: {ruta_archivo}")

    # --- 2. Preparación de Datos ---
    # Trabajamos sobre una copia única de las combinaciones causa-texto
    base_general_filtrada_pd_ciie = base_general_filtrada_pd[["cau221rx_std", "cau_cie10"]].drop_duplicates().copy()

    # Extraer códigos y crear versiones limpias para comparar
    base_general_filtrada_pd_ciie["cie_code"] = base_general_filtrada_pd_ciie["cau_cie10"].apply(extract_cie_code)
    base_general_filtrada_pd_ciie["text_clean"] = base_general_filtrada_pd_ciie["cau_cie10"].apply(get_clean_text)

    # --- 3. Mapeo Inicial (Solo para filas que SÍ tienen código) ---
    # Esto agrupa variantes como "DIABETES A10" y "DIABETES TIPO 2 A10" bajo un mismo nombre
    mapping_with_code = {}
    df_with_code = base_general_filtrada_pd_ciie[base_general_filtrada_pd_ciie["cie_code"].notna()].copy()

    for (cau_val, cie_val), subdf in df_with_code.groupby(["cau221rx_std", "cie_code"]):
        values = subdf["cau_cie10"].dropna().unique()
        if len(values) > 0:
            canonical = max(values, key=len)
            for v in values:
                mapping_with_code[(cau_val, v)] = canonical

    # Aplicar este primer nivel de estandarización
    base_general_filtrada_pd_ciie["cau_cie10_temp"] = base_general_filtrada_pd_ciie.apply(
        lambda row: mapping_with_code.get((row["cau221rx_std"], row["cau_cie10"]), row["cau_cie10"]), 
        axis=1
    )

    # --- 4. Preparación de Candidatos para filas SIN código ---
    # Creamos un diccionario: { causa: { "texto limpio": "Texto Original Con Código" } }
    candidates_by_cau = {}
    df_candidatos = base_general_filtrada_pd_ciie[base_general_filtrada_pd_ciie["cie_code"].notna()]

    for cau, group in df_candidatos.groupby("cau221rx_std"):
        # Usamos text_clean para comparar, pero apuntamos a cau_cie10_temp (que tiene el código)
        candidates_by_cau[cau] = dict(zip(group["text_clean"], group["cau_cie10_temp"]))

    # --- 5. Función de Mapeo para Filas sin Código ---
    def map_no_code_row(row):
        # Si ya tiene código, mantenemos el valor procesado en el paso 3
        if pd.notna(row["cie_code"]):
            return row["cau_cie10_temp"]
        
        cau = row["cau221rx_std"]
        text_to_match = row["text_clean"]
        
        if cau not in candidates_by_cau or not text_to_match:
            return row["cau_cie10_temp"]
        
        choices_dict = candidates_by_cau[cau]
        choices_clean = list(choices_dict.keys())
        
        # Comparar el texto sin código contra los candidatos (también sin código)
        res = process.extractOne(
            text_to_match, 
            choices_clean, 
            scorer=fuzz.token_sort_ratio
        )
        
        if res:
            match_clean, score, idx = res
            if score >= SIM_THRESHOLD:
                # Si hay match, devolvemos la versión QUE TIENE el código
                return choices_dict[match_clean]
                
        return row["cau_cie10_temp"]

    # --- 6. Aplicación Final y Mapeo al DataFrame Original ---
    # Crear la columna limpia definitiva
    base_general_filtrada_pd_ciie["cau_cie10_std"] = base_general_filtrada_pd_ciie.apply(map_no_code_row, axis=1)

    # Crear el diccionario de mapeo final: { texto_sucio_original: texto_estandarizado_con_cie }
    mapping_final_dict = dict(zip(base_general_filtrada_pd_ciie["cau_cie10"], base_general_filtrada_pd_ciie["cau_cie10_std"]))
    del base_general_filtrada_pd
    del base_general_filtrada_pd_ciie
        #guardar el diccionario para cie10
    mapping_final_dict.update({None:"No definido"})
    # the dictiionary has to be saved in utf-8 encoding add the keys
    with open('mapping_cie10_2.txt', 'w', encoding='utf-8') as f:
        f.write("dictionary = {\n")
        for key, value in mapping_final_dict.items():
            if key is not None:
                f.write(f'\"{key}\": \"{value}\",\n')
            else: 
                f.write(f'{key}: \"{value}\",\n')
        f.write("}")


In [ ]:

# Aplicar al DataFrame original
base_general_filtrada_pd["cau_cie10_std"] = base_general_filtrada_pd["cau_cie10"].map(mapping_final_dict)

print("Proceso completado. Columna 'cau_cie10_std' creada.")

In [ ]:
base_general_filtrada_pd.info()

In [ ]:
base_general_filtrada_cie_10=pl.from_pandas(base_general_filtrada_pd)
del base_general_filtrada_pd
gc.collect()

### Resultados

In [ ]:
base_general_filtrada=base_general_filtrada_cie_10.clone()
base_general_filtrada.select(pl.col("cau_cie10_std").is_null().sum())

In [ ]:
# see wich cau221rx doesnt have a match in cau221rx_dict
cau_cie10_std_not_mapped=base_general_filtrada.filter(base_general_filtrada["cau_cie10_std"].is_null())


In [ ]:
print(f"Códigos CIE-10 sin mapear: {cau_cie10_std_not_mapped['cau_cie10'].unique().to_list()}")

In [ ]:
del base_general_filtrada_cie_10
gc.collect()

In [ ]:
base_general_filtrada.describe()

# MAPEO ubicacion

In [ ]:
# --- 1. Diccionario de Homologación Manual ---
# Esto resuelve los casos de zonas no delimitadas y errores de dedo específicos
correcciones_canton_dict = {
    ("ZONAS NO DELIMITADAS", "LAS GOLONDRINAS"): ("IMBABURA", "COTACACHI"),
    ("ZONAS NO DELIMITADAS", "EL PIEDRERO"): ("GUAYAS", "EL TRIUNFO"),
    ("ZONAS NO DELIMITADAS", "MANGA DEL CURA"): ("MANABI", "EL CARMEN"),
    ("LOJA", "OLEMDO"): ("LOJA", "OLMEDO"),
    ("CANAR","AZOQUES"): ("CANAR", "AZOGUES"),
    ("GUAYAS","EMPALME"): ("GUAYAS", "EL EMPALME"),
    ("GUAYAS","SALITRE (URBINA JADO)"):("GUAYAS","SALITRE"),
    ("SANTO DOMINGO DE LOS TSACHILAS","LA CONDORDIA"):("SANTO DOMINGO DE LOS TSACHILAS","LA CONCORDIA"),
    ("ZAMORA CHINCHIPE","YANTZAZA (YANZATZA)"):("ZAMORA CHINCHIPE","YANTZAZA"),
    ("IMBABURA","COTACAHI"): ("IMBABURA", "COTACACHI"),
    ("GUAYAS","CRNEL. MARCELINO MARIDUENA"):("GUAYAS","CORONEL MARCELINO MARIDUENA"),
    ("NAPO","CARLOS JULIO ARROSEMENA TOLA"):("NAPO","CARLOS JULIO AROSEMENA TOLA"),
    ("GUAYAS","GENERAL ANTONIO ELIZALDE (BUCAY)"):("GUAYAS","GENERAL ANTONIO ELIZALDE"),
    ("GUAYAS","GNRAL. ANTONIO ELIZALDE"):("GUAYAS","GENERAL ANTONIO ELIZALDE"),
    ("GUAYAS","ALFREDO BAQUERIZO MORENO (JUJAN)"):("GUAYAS","ALFREDO BAQUERIZO MORENO")
}

correcciones_parroquia_dict = {
    ("PICHINCHA","QUITO","QUITO"): ("PICHINCHA","QUITO","QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL"),
    ("GUAYAS","GUAYAQUIL","GUAYAQUIL"): ("GUAYAS","GUAYAQUIL","GUAYAQUIL , CABECERA CANTONAL Y CAPITAL PROVINCIAL")}


In [ ]:
def aplicar_correcciones_master_canton(df, sufijo):
    col_prov = f"prov_{sufijo}"
    col_cant = f"cant_{sufijo}"
    
    # 1. Limpieza inicial de strings
    df = df.with_columns([
        pl.col(col_prov).str.strip_chars().str.to_uppercase(),
        pl.col(col_cant).str.strip_chars().str.to_uppercase()
    ])
    
    # 2. Aplicar la lógica del diccionario
    for (p_old, c_old), (p_new, c_new) in correcciones_canton_dict.items():
        df = df.with_columns([
            pl.when((pl.col(col_prov) == p_old) & (pl.col(col_cant) == c_old))
            .then(pl.lit(p_new))
            .otherwise(pl.col(col_prov))
            .alias(col_prov),
            
            pl.when((pl.col(col_prov) == p_old) & (pl.col(col_cant) == c_old))
            .then(pl.lit(c_new))
            .otherwise(pl.col(col_cant))
            .alias(col_cant)
        ])
    return df

# Aplicar a la base original
base_general_filtrada = aplicar_correcciones_master_canton(base_general_filtrada, "res")
base_general_filtrada = aplicar_correcciones_master_canton(base_general_filtrada, "ubi")


def aplicar_correcciones_master_parroquia(df, sufijo):
    col_prov = f"prov_{sufijo}"
    col_cant = f"cant_{sufijo}"
    col_parr = f"parr_{sufijo}"
    
    # 1. Limpieza inicial de strings
    df = df.with_columns([
        pl.col(col_prov).str.strip_chars().str.to_uppercase(),
        pl.col(col_cant).str.strip_chars().str.to_uppercase(),
        pl.col(col_parr).str.strip_chars().str.to_uppercase()
    ])
    
    for (p_old, c_old, par_old), (p_new, c_new, par_new) in correcciones_parroquia_dict.items():
        # Definimos la condición de coincidencia para las 3 columnas
        condicion = (
            (pl.col(col_prov) == p_old) & 
            (pl.col(col_cant) == c_old) & 
            (pl.col(col_parr) == par_old)
        )
        
        df = df.with_columns([
            # Corregir Provincia
            pl.when(condicion)
            .then(pl.lit(p_new))
            .otherwise(pl.col(col_prov))
            .alias(col_prov),
            
            # Corregir Cantón
            pl.when(condicion)
            .then(pl.lit(c_new))
            .otherwise(pl.col(col_cant))
            .alias(col_cant),

            # Corregir Parroquia
            pl.when(condicion)
            .then(pl.lit(par_new))
            .otherwise(pl.col(col_parr))
            .alias(col_parr)
        ])
        
    return df

# Aplicar a la base original
base_general_filtrada = aplicar_correcciones_master_canton(base_general_filtrada, "res")
base_general_filtrada = aplicar_correcciones_master_canton(base_general_filtrada, "ubi")

# Aplicar a la base original
base_general_filtrada = aplicar_correcciones_master_parroquia(base_general_filtrada, "res")
base_general_filtrada = aplicar_correcciones_master_parroquia(base_general_filtrada, "ubi")


## MAPEO PARROQUIAS

In [ ]:
inec_parroquias=pd.read_csv("../../data/parroquias_cantones_inec.csv")

In [ ]:
columns_to_strip_inec=["canton","parroquia","provincia"]
for column in columns_to_strip_inec:
    inec_parroquias[column] = inec_parroquias[column].str.strip()

In [ ]:
inec_parroquias=inec_parroquias[["provincia","canton","parroquia","code_parroquia","sri_parroquia"]].drop_duplicates()

### Mapeo parroquias Hospitales

In [ ]:

# --- 2. Función de Limpieza Extra ---
def clean_geo_text(text):
    if pd.isna(text): return ""
    text = str(text).upper()
   

    # Elimina todo lo que esté entre paréntesis: "ALAQUES (ALAQUEZ)" -> "ALAQUES"
    text=text.replace("(", "").replace(")", "")

    stop_words = [
        "DISTRITO METROPOLITANO DE", "D.M.", 
        "CABECERA CANTONAL", "PARROQUIA URBANA", 
        "PARROQUIA RURAL", "CENTRO POBLADO",
        "CANTON", "AREA URBANA", "PARROQUIA"
    ]
    for word in stop_words:
        text = text.replace(word, "")

    # Elimina caracteres especiales y deja solo letras y espacios
    text = re.sub(r'[^A-ZÁÉÍÓÚÑ\s]', '', text)
    return " ".join(text.split()) # Elimina espacios extra

# --- 3. Función de Fuzzy Match Mejorada ---
def find_best_location(row, reference_df, threshold=80,sufix="ubi"):
    prov_actual = row[f"prov_{sufix}"]
    candidates = reference_df[reference_df[f"prov_{sufix}"] == prov_actual].copy()
    
    if candidates.empty:
        return pd.Series([None, None], index=["code_parroquia", "sri_parroquia"])

    # Limpiamos el texto de entrada (el de tu base)
    # Comparamos Cantón + Parroquia limpios
    query_canton = clean_geo_text(row[f'cant_{sufix}'])
    query_parr = clean_geo_text(row[f'parr_{sufix}'])
    query = f"{query_canton} {query_parr}"
    
    # Preparamos la lista de opciones (limpias también)
    choices = (candidates[f"cant_{sufix}"].apply(clean_geo_text) + " " + 
            candidates[f"parr_{sufix}"].apply(clean_geo_text)).tolist()
    
    # Usamos token_set_ratio: es excelente para "ALAQUES" vs "ALAQUEZ" 
    # y maneja mucho mejor las diferencias ortográficas pequeñas.
    res = process.extractOne(
        query, 
        choices, 
        scorer=fuzz.token_set_ratio 
    )
    
    if res:
        match_str, score, idx = res
        if score >= threshold:
            matched_row = candidates.iloc[idx]
            return pd.Series([matched_row["code_parroquia"], matched_row["sri_parroquia"]], 
                            index=["code_parroquia", "sri_parroquia"])
    
    return pd.Series([None, None], index=["code_parroquia", "sri_parroquia"])


In [ ]:

#
print("Iniciando Fuzzy Match de ubicaciones...")

# --- 2. Preparación de tu base ---
inec_parroquias_ubi =inec_parroquias.rename(columns={"provincia":"prov_ubi","canton":"cant_ubi","parroquia":"parr_ubi"})

base_general_filtrada_ubi=base_general_filtrada.select(territorio_ubi).unique()
print(f"Parroquias a matchear: {base_general_filtrada_ubi.shape[0]}")
base_general_filtrada_ubi_pd=base_general_filtrada_ubi.to_pandas()
for col in territorio_ubi:
    base_general_filtrada_ubi_pd[col] = base_general_filtrada_ubi_pd[col].str.strip().str.upper()


# 1. Limpiamos el catálogo del INEC antes del merge
catálogo_para_merge = (
    inec_parroquias_ubi
    .sort_values(by="sri_parroquia", ascending=False, na_position='last')
    .drop_duplicates(subset=["prov_ubi", "cant_ubi", "parr_ubi"], keep="first")
)


# A. Merge Exacto primero
base_final_ubi = base_general_filtrada_ubi_pd.merge(
    catálogo_para_merge[["prov_ubi", "cant_ubi", "parr_ubi","code_parroquia"]], 
    on=["prov_ubi", "cant_ubi", "parr_ubi"], 
    how="left"
)

mask_missing = base_final_ubi["code_parroquia"].isna()
print(f"Filas encontradas exactamente: {(~mask_missing).sum()}")
print(f"Filas para corregir con Fuzzy Match: {mask_missing.sum()}")

# B. Aplicar Fuzzy Match solo donde falló el exacto
if mask_missing.any():
    # El uso de result_type="expand" asegura que el resultado se asigne correctamente a las dos columnas
    fuzzy_results = base_final_ubi[mask_missing].apply(
        lambda r: find_best_location(r, catálogo_para_merge,sufix="ubi"), 
        axis=1
    )
    
    base_final_ubi.loc[mask_missing, ["code_parroquia"]] = fuzzy_results

# --- 5. Resultados ---
# Limpiar tipos de datos si es necesario (códigos a string/int)
base_final_ubi["code_parroquia"] = base_final_ubi["code_parroquia"].astype(str).replace("None", pd.NA)

# Mostrar algunas correcciones para validar
print("Mapeo de ubicación terminado.")
print(f"Filas totales: {base_final_ubi.shape[0]}")
print(f"Filas matcheadas: {base_final_ubi['code_parroquia'].notna().sum()}")
print(f"Códigos parroquia únicos encontrados: {base_final_ubi['code_parroquia'].nunique()}")

### Mapeo Parroquias residencia

In [ ]:

print("Iniciando Fuzzy Match de ubicaciones...")
inec_parroquias_res=inec_parroquias.rename(columns={"provincia":"prov_res","canton":"cant_res","parroquia":"parr_res"})

# 1. Limpiamos el catálogo del INEC antes del merge
catálogo_para_merge = (
    inec_parroquias_res
    .sort_values(by="sri_parroquia", ascending=False, na_position='last')
    .drop_duplicates(subset=["prov_res", "cant_res", "parr_res"], keep="first")
)


base_general_filtrada_res=base_general_filtrada.select(territorio_res).unique()
print(f"Parroquias a matchear: {base_general_filtrada_res.shape[0]}")
base_general_filtrada_res_pd=base_general_filtrada_res.to_pandas()
# --- 2. Preparación de tu base ---

for col in territorio_res:
    base_general_filtrada_res_pd[col] = base_general_filtrada_res_pd[col].str.strip().str.upper()


# --- 4. Ejecución del Proceso ---

# A. Merge Exacto primero
base_final_res = base_general_filtrada_res_pd.merge(
    catálogo_para_merge[["prov_res", "cant_res", "parr_res","code_parroquia"]], 
    on=["prov_res", "cant_res", "parr_res"], 
    how="left"
)

mask_missing = base_final_res["code_parroquia"].isna()
print(f"Filas encontradas exactamente: {(~mask_missing).sum()}")
print(f"Filas para corregir con Fuzzy Match: {mask_missing.sum()}")

# B. Aplicar Fuzzy Match solo donde falló el exacto
if mask_missing.any():
    # El uso de result_type="expand" asegura que el resultado se asigne correctamente a las dos columnas
    fuzzy_results = base_final_res[mask_missing].apply(
        lambda r: find_best_location(r, catálogo_para_merge,sufix="res"), 
        axis=1
    )
    
    base_final_res.loc[mask_missing, ["code_parroquia"]] = fuzzy_results

# --- 5. Resultados ---
# Limpiar tipos de datos si es necesario (códigos a string/int)
base_final_res["code_parroquia"] = base_final_res["code_parroquia"].astype(str).replace("None", pd.NA)

# Mostrar algunas correcciones para validar
print("Mapeo de ubicación terminado.")
print(f"Filas totales: {base_final_res.shape[0]}")
print(f"Filas matcheadas: {base_final_res['code_parroquia'].notna().sum()}")
#poner el numero de code parroquia unico
print(f"Códigos parroquia únicos encontrados: {base_final_res['code_parroquia'].nunique()}")

### Revision duplicados


In [ ]:
# Ver qué combinaciones están repetidas en el catálogo del INEC
duplicados_inec = catálogo_para_merge[
    catálogo_para_merge.duplicated(subset=["prov_res", "cant_res", "parr_res"], keep=False)
]
print("Filas duplicadas en el catálogo INEC que causan la explosión:")
print(duplicados_inec)

In [ ]:
# Para Residencia
base_final_res["key_text"] = (
    base_final_res["prov_res"].apply(clean_geo_text) + "_" + 
    base_final_res["cant_res"].apply(clean_geo_text) + "_" + 
    base_final_res["parr_res"].apply(clean_geo_text)
)

# Para Ubicación
base_final_ubi["key_text"] = (
    base_final_ubi["prov_ubi"].apply(clean_geo_text) + "_" + 
    base_final_ubi["cant_ubi"].apply(clean_geo_text) + "_" + 
    base_final_ubi["parr_ubi"].apply(clean_geo_text)
)

In [ ]:
# El equivalente a is_duplicated() de Polars es duplicated(keep=False)
base_final_ubi_duplicates = base_final_ubi[base_final_ubi.duplicated(subset=["code_parroquia"], keep=False)]

base_final_res_duplicates = base_final_res[base_final_res.duplicated(subset=["code_parroquia"], keep=False)]



In [ ]:
#review duplicados
print("Duplicados en base_final_ubi:")
print(base_final_ubi_duplicates[["prov_ubi","cant_ubi","parr_ubi","code_parroquia"]].sort_values("code_parroquia"))


In [ ]:

print("\nDuplicados en base_final_res:")
print(base_final_res_duplicates[["prov_res","cant_res","parr_res","code_parroquia"]].sort_values("code_parroquia"))


### Mapeo Shape file Parroquia

In [ ]:

# --- FUNCIÓN FUZZY CORREGIDA ---
def find_best_conali_match(row, conali_df, threshold=80, sufix="res"):
    prov_actual = row[f"prov_{sufix}"]
    # Filtramos por prov_clean que ya está en el df
    candidates = conali_df[conali_df["prov_clean"] == clean_geo_text(prov_actual)].copy()
    
    if candidates.empty:
        return None

    query_canton = clean_geo_text(row[f'cant_{sufix}'])
    query_parr = clean_geo_text(row[f'parr_{sufix}'])
    query = f"{query_canton} {query_parr}"
    
    # Usamos las columnas ya limpias del shapefile
    choices = (candidates["cant_clean"] + " " + candidates["parr_clean"]).tolist()
    
    res = process.extractOne(query, choices, scorer=fuzz.token_set_ratio)
    
    if res:
        match_str, score, idx = res
        if score >= threshold:
            return candidates.index[idx]
    return None


In [ ]:
import geopandas as gpd
cantones = gpd.read_file("../../data/organizacion-territorial-cantonal/ORGANIZACION_TERRITORIAL_CANTONAL.shp")
conali_parroquias_shape=gpd.read_file("../../data/LIMITE_PARROQUIAL_CONALI_CNE_2022/LIMITE_PARROQUIAL_CONALI_CNE_2022/LIMITE_PARROQUIAL_CONALI_CNE_2022.shp")

In [ ]:
conali_parroquias=conali_parroquias_shape[["PROVINCIA", "CANTON","PARROQUIA","CODPAR"]]

In [ ]:
conali_parroquias.columns

In [ ]:
# --- 1. PREPARACIÓN DE DICCIONARIO DE CÓDIGOS (std_parroquias) ---
# Limpiamos el código viejo para que coincida con el CODPAR del shapefile
std_parroquias = pd.read_csv("std_parroquias.csv")
std_parroquias["PARROQUIA_CODIGO_OLD"] = std_parroquias["PARROQUIA_CODIGO_OLD"].astype(str).str.replace(".0", "", regex=False).str.strip().str.zfill(4)
mapeo_old_to_new = dict(zip(std_parroquias["PARROQUIA_CODIGO_OLD"], std_parroquias["PARROQUIA_CODIGO"]))

conali_parroquias["code_parroquia"] = conali_parroquias["CODPAR"].astype(str).str.strip().map(mapeo_old_to_new)

In [ ]:
# Esto te mostrará qué códigos están repetidos en el shapefile
culpables_conali = conali_parroquias[conali_parroquias.duplicated(subset="code_parroquia", keep=False)]
print(culpables_conali.sort_values("code_parroquia"))

In [ ]:
# Definir las columnas correctas del shapefile CONALI
# Según tu inspección: 'PROVINCIA', 'CANTON', 'PARROQUIA'
conali_parroquias["prov_clean"] = conali_parroquias["PROVINCIA"].apply(clean_geo_text)
conali_parroquias["cant_clean"] = conali_parroquias["CANTON"].apply(clean_geo_text)
conali_parroquias["parr_clean"] = conali_parroquias["PARROQUIA"].apply(clean_geo_text)

# Crear una llave única de texto para el join
conali_parroquias["key_text"] = (
    conali_parroquias["prov_clean"] + "_" + 
    conali_parroquias["cant_clean"] + "_" + 
    conali_parroquias["parr_clean"]
)

In [ ]:
# Paso 1: Intentar por Código INEC (Merge Exacto)
# Unimos tu base_final_res con el shapefile usando code_parroquia
print(f"Total de registros antes del merge: {len(base_final_res)}")
print("Intentando match por código INEC...")


# 1. Filtramos: Solo registros que TIENEN código y eliminamos duplicados de esos códigos
conali_para_codigo = (
    conali_parroquias
    .dropna(subset=["code_parroquia"]) # <--- CRUCIAL: Elimina los NaNs que causan la explosión
    .drop_duplicates(subset=["code_parroquia"])
)


base_geo_res = base_final_res.merge(
    conali_para_codigo[["code_parroquia", "CODPAR"]],
    left_on="code_parroquia",
    right_on="code_parroquia",
    how="left")
# Paso 2: Fallback por Texto Exacto
print(f"Registros sin geometría después del merge por código: {base_geo_res['CODPAR'].isna().sum()}")

mask_no_geo = base_geo_res["CODPAR"].isna()
if mask_no_geo.any():
    print(f"Paso 2: Intentando match por texto exacto para {mask_no_geo.sum()} filas...")
    base_geo_res.loc[mask_no_geo, "key_text"] = (
        base_geo_res.loc[mask_no_geo, "prov_res"].apply(clean_geo_text) + "_" +
        base_geo_res.loc[mask_no_geo, "cant_res"].apply(clean_geo_text) + "_" +
        base_geo_res.loc[mask_no_geo, "parr_res"].apply(clean_geo_text)
    )
    dict_geo_text = dict(zip(conali_parroquias["key_text"], conali_parroquias["CODPAR"]))
    base_geo_res.loc[mask_no_geo, "CODPAR"] = base_geo_res.loc[mask_no_geo, "key_text"].map(dict_geo_text)

# Paso 3: Fuzzy Match fallback
print(f"Registros sin geometría después del match por texto exacto: {base_geo_res['CODPAR'].isna().sum()}")

mask_missing = base_geo_res["CODPAR"].isna()
if mask_missing.any():
    print(f"Haciendo Fuzzy Match para {mask_missing.sum()} geometrías de residencia...")
    idx_matches = base_geo_res[mask_missing].apply(
        lambda r: find_best_conali_match(r, conali_parroquias, sufix="res"), axis=1
    )
    
    for row_idx, conali_idx in idx_matches.dropna().items():
        base_geo_res.at[row_idx, "CODPAR"] = conali_parroquias.at[conali_idx, "CODPAR"]


print(f"Total de registros: {len(base_geo_res)}")
print(f"Registros SIN geometría: {base_geo_res['CODPAR'].isna().sum()}")

print("Proceso terminado.")

In [ ]:
# Paso 1: Intentar por Código INEC (Merge Exacto)
# Unimos tu base_final_res con el shapefile usando code_parroquia
print(f"Total de registros antes del merge: {len(base_final_ubi)}")
print("Intentando match por código INEC...")

# 1. Filtramos: Solo registros que TIENEN código y eliminamos duplicados de esos códigos
conali_para_codigo = (
    conali_parroquias
    .dropna(subset=["code_parroquia"]) # <--- CRUCIAL: Elimina los NaNs que causan la explosión
    .drop_duplicates(subset=["code_parroquia"])
)

base_geo_ubi = base_final_ubi.merge(
    conali_para_codigo[["code_parroquia", "CODPAR"]],
    left_on="code_parroquia",
    right_on="code_parroquia",
    how="left")
# Paso 2: Fallback por Texto Exacto
print(f"Registros sin geometría después del merge por código: {base_geo_ubi['CODPAR'].isna().sum()}")

mask_no_geo = base_geo_ubi["CODPAR"].isna()
if mask_no_geo.any():
    print(f"Paso 2: Intentando match por texto exacto para {mask_no_geo.sum()} filas...")
    base_geo_ubi.loc[mask_no_geo, "key_text"] = (
        base_geo_ubi.loc[mask_no_geo, "prov_ubi"].apply(clean_geo_text) + "_" +
        base_geo_ubi.loc[mask_no_geo, "cant_ubi"].apply(clean_geo_text) + "_" +
        base_geo_ubi.loc[mask_no_geo, "parr_ubi"].apply(clean_geo_text)
    )
    dict_geo_text = dict(zip(conali_parroquias["key_text"], conali_parroquias["CODPAR"]))
    base_geo_ubi.loc[mask_no_geo, "CODPAR"] = base_geo_ubi.loc[mask_no_geo, "key_text"].map(dict_geo_text)

# Paso 3: Fuzzy Match fallback
print(f"Registros sin geometría después del match por texto exacto: {base_geo_res['CODPAR'].isna().sum()}")

mask_missing = base_geo_ubi["CODPAR"].isna()
if mask_missing.any():
    print(f"Haciendo Fuzzy Match para {mask_missing.sum()} geometrías de residencia...")
    idx_matches = base_geo_ubi[mask_missing].apply(
        lambda r: find_best_conali_match(r, conali_parroquias, sufix="ubi"), axis=1
    )
    
    for row_idx, conali_idx in idx_matches.dropna().items():
        base_geo_ubi.at[row_idx, "CODPAR"] = conali_parroquias.at[conali_idx, "CODPAR"]

print(f"Total de registros: {len(base_geo_ubi)}")
print(f"Registros SIN geometría: {base_geo_ubi['CODPAR'].isna().sum()}")
print("Proceso terminado.")

In [ ]:

# 4. Convertir a GeoDataFrame final y procesar coordenadas
#ver el de parroquias
print(conali_parroquias_shape.crs)
conali_parroquias_shape = conali_parroquias_shape.to_crs(epsg=4326)
print(conali_parroquias_shape.crs)
conali_parroquias_shape["centroid"] = conali_parroquias_shape.geometry.centroid
conali_parroquias_shape["lat_parroquia"] = conali_parroquias_shape.centroid.y
conali_parroquias_shape["lon_parroquia"] = conali_parroquias_shape.centroid.x

In [ ]:
base_geo_res=base_geo_res.merge(conali_parroquias_shape[["CODPAR","lat_parroquia","lon_parroquia"]], on="CODPAR", how="left").rename(columns={"lat_parroquia":"lat_res","lon_parroquia":"lon_res"}).drop(columns=["CODPAR"])
base_geo_ubi=base_geo_ubi.merge(conali_parroquias_shape[["CODPAR","lat_parroquia","lon_parroquia"]], on="CODPAR", how="left").rename(columns={"lat_parroquia":"lat_ubi","lon_parroquia":"lon_ubi"}).drop(columns=["CODPAR"])


In [ ]:
base_geo_res_pl=pl.from_pandas(base_geo_res)
base_geo_ubi_pl=pl.from_pandas(base_geo_ubi)
del base_geo_res
del base_geo_ubi


In [ ]:

base_general_filtrada=base_general_filtrada.join(base_geo_res_pl,on=["prov_res","cant_res","parr_res"],how="left")
base_general_filtrada=base_general_filtrada.join(base_geo_ubi_pl,on=["prov_ubi","cant_ubi","parr_ubi"],how="left")

In [ ]:
print(base_general_filtrada.select(pl.col("lat_ubi","lat_res").is_null().sum()))

In [ ]:
# review from parr_ubi and parr_res cuantos son los que tienen más null en lat_ubi and lat_res
# Filtrar las filas donde 'lat_res' es nulo y mostrar el valor de 'parr_res'

# Filtrar las filas donde 'lat_res' es nulo y luego agrupar por 'parr_res'
result_res = base_general_filtrada.filter(pl.col("lat_res").is_null()) \
    .group_by("prov_res","cant_res","parr_res") \
    .agg(pl.len().alias("null_count"))\
    .sort("null_count",descending=True)

# Mostrar el resultado
print(result_res)



In [ ]:
# Filtrar las filas donde 'lat_res' es nulo y luego agrupar por 'parr_res'
result_ubi = base_general_filtrada.filter(pl.col("lat_ubi").is_null()) \
    .group_by("prov_ubi","cant_ubi","parr_ubi") \
    .agg(pl.len().alias("null_count")) \
    .sort("null_count",descending=True)
# Mostrar el resultado
print(result_ubi)




In [ ]:


base_general_filtrada=base_general_filtrada.drop(["key_text_right","key_text"])

In [ ]:
# quitar las columnas key_text, key_text_right. renombrar code_parroquia a code_parroquia_res y code_parroquia_right a code_parroquia_ubi
base_general_filtrada=base_general_filtrada.with_columns([
        pl.col("code_parroquia").alias("code_parr_res"),
        pl.col("code_parroquia_right").alias("code_parr_ubi")
    ])



In [ ]:

#base_general_filtrada=base_general_filtrada.drop(["cau221rx","cau_cie10"])
base_general_filtrada=base_general_filtrada.drop(["code_parroquia_right","code_parroquia"])

In [ ]:
base_general_filtrada.describe()

## Mapeo Cantones

A diferencia que en parroquia donde habia un mapeo de parroquia sri a parroquia inec. Aqui tenemos que tomar directamente el nombre de canton y provincia y hacer el mapeo.



In [ ]:

# --- Función de Limpieza (Se mantiene igual, es robusta) ---
def clean_geo_text(text):
    if pd.isna(text): return ""
    text = str(text).upper()
    
    # Elimina paréntesis y contenido
    text = re.sub(r'\([^)]*\)', '', text)
    
    stop_words = [
        "DISTRITO METROPOLITANO DE", "D.M.", 
        "CABECERA CANTONAL", "CANTON", "AREA URBANA"
    ]
    for word in stop_words:
        text = text.replace(word, "")
    
    text = re.sub(r'[^A-ZÁÉÍÓÚÑ\s]', '', text)
    return " ".join(text.split())

# --- Nueva Función de Fuzzy Match para Cantones ---
def find_best_canton(row, reference_df, threshold=85, col_canton="cant_res", col_prov="prov_res"):
    prov_actual = row[col_prov]
    # Filtramos candidatos por provincia para no matchear "Manta" en una provincia que no es
    candidates = reference_df[reference_df["provincia"] == prov_actual].copy()
    
    if candidates.empty:
        return None

    query_canton = clean_geo_text(row[col_canton])
    choices = candidates["canton"].apply(clean_geo_text).tolist()
    
    # Usamos extractOne para encontrar la mejor coincidencia
    res = process.extractOne(
        query_canton, 
        choices, 
        scorer=fuzz.token_set_ratio
    )
    
    if res:
        match_str, score, idx = res
        if score >= threshold:
            return candidates.iloc[idx]["code_canton"]
    
    return None

### Mapeo Canton residencia



In [ ]:
# 1. Preparar Catálogo de Referencia (INEC)
inec_parroquias=pd.read_csv("../../data/parroquias_cantones_inec.csv")
cat_cantones = inec_parroquias[["provincia", "canton", "code_canton"]].drop_duplicates()

base_general_filtrada_res=base_general_filtrada.select(['prov_res','cant_res']).unique()


base_general_filtrada_res_pd=base_general_filtrada_res.to_pandas()

for col in ['prov_res','cant_res']:
    base_general_filtrada_res_pd[col] = base_general_filtrada_res_pd[col].str.strip().str.upper()


print(f"Cantones únicos a procesar: {len(base_general_filtrada_res_pd)}")

# A. Intento de Merge Exacto
base_general_filtrada_res_pd = base_general_filtrada_res_pd.merge(
    cat_cantones, 
    left_on=["prov_res", "cant_res"], 
    right_on=["provincia", "canton"], 
    how="left"
)

# B. Aplicar Fuzzy Match a los nulos
mask_missing = base_general_filtrada_res_pd["code_canton"].isna()
print(f"Fallas en match exacto: {mask_missing.sum()}")

if mask_missing.any():
    base_general_filtrada_res_pd.loc[mask_missing, "code_canton"] = base_general_filtrada_res_pd[mask_missing].apply(
        lambda r: find_best_canton(r, cat_cantones, col_canton="cant_res", col_prov="prov_res"), 
        axis=1
    )

# 3. Mapear los resultados de vuelta a tu msp_data original
# Creamos un diccionario de mapeo: (prov, cant) -> codigo
mapping_dict = base_general_filtrada_res_pd.set_index(["prov_res", "cant_res"])["code_canton"].to_dict()

base_general_filtrada_res_pd["code_cant_res"] = base_general_filtrada_res_pd.set_index(["prov_res", "cant_res"]).index.map(mapping_dict)

print("Proceso finalizado.")
print(f"Total matcheado: {base_general_filtrada_res_pd['code_cant_res'].notna().sum()} de {len(base_general_filtrada_res_pd)}")
base_general_filtrada_res_pd.drop(columns=["code_canton","provincia","canton"],inplace=True)

Consideraciones:
- LAS GOLONDRINAS: Es ahora parte de canton "COTACACHI" de IMBABURA
- EL PIEDRERO: es parte del canton "EL TRIUNFO" de GUAYAS
- MANGA DEL CURA: es parte del canton "EL CARMEN" de "MANABI"
- Exterior: 
- OLEMDO: es en realidad "OLMEDO" de LOJA

### Mapeo Canton ubicación

In [ ]:

base_general_filtrada_ubi=base_general_filtrada.select(['prov_ubi','cant_ubi']).unique()


base_general_filtrada_ubi_pd=base_general_filtrada_ubi.to_pandas()

for col in ['prov_ubi','cant_ubi']:
    base_general_filtrada_ubi_pd[col] = base_general_filtrada_ubi_pd[col].str.strip().str.upper()


print(f"Cantones únicos a procesar: {len(base_general_filtrada_ubi_pd)}")

# A. Intento de Merge Exacto
base_general_filtrada_ubi_pd = base_general_filtrada_ubi_pd.merge(
    cat_cantones, 
    left_on=["prov_ubi", "cant_ubi"], 
    right_on=["provincia", "canton"], 
    how="left"
)

# B. Aplicar Fuzzy Match a los nulos
mask_missing = base_general_filtrada_ubi_pd["code_canton"].isna()
print(f"Fallas en match exacto: {mask_missing.sum()}")

if mask_missing.any():
    base_general_filtrada_ubi_pd.loc[mask_missing, "code_canton"] = base_general_filtrada_ubi_pd[mask_missing].apply(
        lambda r: find_best_canton(r, cat_cantones, col_canton="cant_ubi", col_prov="prov_ubi"), 
        axis=1
    )

# 3. Mapear los ubiultados de vuelta a tu msp_data original
# Creamos un diccionario de mapeo: (prov, cant) -> codigo
mapping_dict = base_general_filtrada_ubi_pd.set_index(["prov_ubi", "cant_ubi"])["code_canton"].to_dict()

base_general_filtrada_ubi_pd["code_cant_ubi"] = base_general_filtrada_ubi_pd.set_index(["prov_ubi", "cant_ubi"]).index.map(mapping_dict)

print("Proceso finalizado.")
print(f"Total matcheado: {base_general_filtrada_ubi_pd['code_cant_ubi'].notna().sum()} de {len(base_general_filtrada_ubi_pd)}")
base_general_filtrada_ubi_pd.drop(columns=["code_canton","provincia","canton"],inplace=True)

### Mapeo Shape file cantones


In [ ]:
cantones = gpd.read_file("../../data/organizacion-territorial-cantonal/ORGANIZACION_TERRITORIAL_CANTONAL.shp")

# Ver el CRS actual
print(cantones.crs)
# Convertir a lat/lon (EPSG:4326)
cantones = cantones.to_crs(epsg=4326)
cantones["centroid"] = cantones.geometry.centroid
cantones["lat_canton"] = cantones.centroid.y
cantones["lon_canton"] = cantones.centroid.x
cantones["DPA_CANTON"] = "EC" + cantones["DPA_CANTON"].astype(str)
cantones.rename(columns={"DPA_CANTON":"code_canton"},inplace=True)


In [ ]:
base_general_filtrada_ubi_pd = base_general_filtrada_ubi_pd.merge(
        cantones[["code_canton", "lat_canton", "lon_canton"]],
        left_on="code_cant_ubi",
        right_on="code_canton",
        how="left"
    )

base_general_filtrada_res_pd = base_general_filtrada_res_pd.merge(
        cantones[["code_canton", "lat_canton", "lon_canton"]],
        left_on="code_cant_res",
        right_on="code_canton",
        how="left"
)
base_general_filtrada_res_pd.drop(columns=["code_canton"],inplace=True)

base_general_filtrada_ubi_pd.drop(columns=["code_canton"],inplace=True)

In [ ]:
print("Columnas  base_general_filtrada_res_pd:",base_general_filtrada_res_pd.columns)
print("Columnas  base_general_filtrada_ubi_pd:",base_general_filtrada_ubi_pd.columns)

### Join con base general

Vamos a colocar el codigo del canton y si está vacia la latitud longitud, la rellenamos con las que vinen del canton

In [ ]:
# 2. Convertir Pandas a Polars y renombrar para evitar colisiones
res_ref = pl.from_pandas(base_general_filtrada_res_pd).rename({
    "lat_canton": "lat_ref_res", 
    "lon_canton": "lon_ref_res"
})

ubi_ref = pl.from_pandas(base_general_filtrada_ubi_pd).rename({
    "lat_canton": "lat_ref_ubi", 
    "lon_canton": "lon_ref_ubi"
})

# 3. Realizar los Joins
base_general_filtrada = base_general_filtrada.join(
    res_ref, on=["prov_res", "cant_res"], how="left"
).join(
    ubi_ref, on=["prov_ubi", "cant_ubi"], how="left"
)

In [ ]:
# 3. Cálculo de Registros Llenados (Antes de Coalesce)
# Contamos dónde el original era null pero la referencia tenía dato
conteo_llenados = base_general_filtrada.select([
    (pl.col("lat_res").is_null() & pl.col("lat_ref_res").is_not_null()).sum().alias("res_lat_llenos"),
    (pl.col("lat_ubi").is_null() & pl.col("lat_ref_ubi").is_not_null()).sum().alias("ubi_lat_llenos")
]).to_dicts()[0]

# 4. Lógica de Exterior y Coalesce de Coordenadas
base_general_filtrada = base_general_filtrada.with_columns([
    # Manejo de Códigos de Cantón (Incluyendo Exterior)
    pl.when(pl.col("prov_res") == "EXTERIOR")
    .then(pl.lit("999999"))
    .otherwise(pl.col("code_cant_res"))
    .alias("code_cant_res"),

    # Relleno de coordenadas por jerarquía (Original -> Cantón)
    pl.coalesce(["lat_res", "lat_ref_res"]).alias("lat_res"),
    pl.coalesce(["lon_res", "lon_ref_res"]).alias("lon_res"),
    pl.coalesce(["lat_ubi", "lat_ref_ubi"]).alias("lat_ubi"),
    pl.coalesce(["lon_ubi", "lon_ref_ubi"]).alias("lon_ubi")
])

# 5. Limpieza Final de columnas auxiliares
base_general_filtrada = base_general_filtrada.drop([
    "lat_ref_res", "lon_ref_res", "lat_ref_ubi", "lon_ref_ubi"
])


In [ ]:

# --- Reporte de Resultados ---
print("--- Reporte de Georreferenciación ---")
print(f"Registros de residencia geocodificados por cantón: {conteo_llenados['res_lat_llenos']}")
print(f"Registros de ubicación geocodificados por cantón:  {conteo_llenados['ubi_lat_llenos']}")
print(f"Total registros 'EXTERIOR' marcados: {(base_general_filtrada['code_cant_res'] == '999999').sum()}")

In [ ]:
# En Polars se usa null_count()
null_count=base_general_filtrada.null_count()
null_count

In [ ]:
base_general_filtrada.describe()

### Afinar coordenadas de algunas parroquias



In [ ]:

# Tu diccionario de coordenadas específicas
new_cords_parroquia_dict = {
    ("PICHINCHA", "QUITO", "QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL"): (-0.19506169985974994, -78.49475894065186),
    ("GUAYAS", "GUAYAQUIL", "GUAYAQUIL , CABECERA CANTONAL Y CAPITAL PROVINCIAL"): (-2.1873106383919407, -79.89663942964654),
    ("GUAYAS", "GUAYAQUIL", "TARQUI"): (-2.129950856875764, -79.89878924477644)
}

def aplicar_nuevas_coordenadas(df, sufijo):
    col_prov = f"prov_{sufijo}"
    col_cant = f"cant_{sufijo}"
    col_parr = f"parr_{sufijo}"
    col_lat = f"lat_{sufijo}"
    col_lon = f"lon_{sufijo}"

    for (p, c, par), (new_lat, new_lon) in new_cords_parroquia_dict.items():
        # Definimos la condición de coincidencia exacta
        condicion = (
            (pl.col(col_prov) == p) & 
            (pl.col(col_cant) == c) & 
            (pl.col(col_parr) == par)
        )
        
        # Aplicamos el cambio a latitud y longitud
        df = df.with_columns([
            pl.when(condicion).then(pl.lit(new_lat)).otherwise(pl.col(col_lat)).alias(col_lat),
            pl.when(condicion).then(pl.lit(new_lon)).otherwise(pl.col(col_lon)).alias(col_lon)
        ])
    
    return df

# Aplicar a residencia y ubicación
base_general_filtrada = aplicar_nuevas_coordenadas(base_general_filtrada, "res")
base_general_filtrada = aplicar_nuevas_coordenadas(base_general_filtrada, "ubi")


>por algua razon las que tienen el tema de cabeza cantonal son las más exactas que las que tieynen unicamente el nombre. 

# Extraer cie 10 especificos

Obesidad E66 (E66.0, E66.01, E66.09, E66.8, E66.9, E66.3, E66.81, E66.811, E66.812, E66.813)

Hipertrigliceridemia: E78, E78.1, E78.3, E78.5, E78.5, E78.6
* Causes

* Diabetes (E10-E14, O24)

gen diabetes = 0
replace diabetes = 1 if causeICD10 == "E11"

* Hypertensive (I10-I15)

gen hyper = 0
replace hyper = 1 if causeICD10 == "I10"

* Myocardial infarction (I21-I22)

gen infarction = 0
replace infarction = 1 if (causeICD10 == "I21" | causeICD10 == "I22")

Esto sí es. Aquí tienes en código de stata lo que estamos usando. En resumen, E11, I10 y I21,I22

In [ ]:
import polars as pl

# 1. Definimos las condiciones como expresiones de Polars
# Usamos (?i) al inicio del regex para que sea case-insensitive (equivalente a case=False)

# Diabetes
cond_diabetes = (
    pl.col("cau_cie10_std").str.contains(r"E1[0-4]|O24") 
    #pl.col("cau_cie10_std").str.contains(r"(?i)DIABETES") |
    #pl.col("cau221rx_std").str.contains(r"(?i)DIABETES")
)

# Obesidad
cond_obesity = (
    pl.col("cau_cie10_std").str.contains(r"E66") |
    pl.col("cau_cie10_std").str.contains(r"(?i)OBES")# |
    #pl.col("cau221rx_std").str.contains(r"(?i)OBES")
)

# Hipertensión (aquí text_enabled era False en tu código)
cond_hyper = pl.col("cau_cie10_std").str.contains(r"I1[0-5]")

# Infartos
cond_infarction = (
    pl.col("cau_cie10_std").str.contains(r"I21|I22") 
    #pl.col("cau_cie10_std").str.contains(r"(?i)INFARTO") |
    #pl.col("cau221rx_std").str.contains(r"(?i)INFARTO")
)

# Hipertrigliceridemia
cond_hiper_trig = (
    pl.col("cau_cie10_std").str.contains(r"E78") #|
    # pl.col("cau_cie10_std").str.contains(r"(?i)HIPERTRI") |
    # pl.col("cau221rx_std").str.contains(r"(?i)HIPERTRI")
)

# 2. Aplicamos todo al DataFrame
base_general_filtrada = base_general_filtrada.with_columns(
    sindrome_metabolico = pl.when(
        cond_diabetes | cond_obesity | cond_hyper | cond_infarction | cond_hiper_trig
    )
    .then(1)
    .otherwise(0)
    .cast(pl.Int8) # Opcional: para que ocupe menos memoria que un Int64
)

# Para ver cuántos casos positivos tienes:
print(base_general_filtrada["sindrome_metabolico"].value_counts())

In [ ]:
resumen_cie10 = (
    base_general_filtrada
    .filter(pl.col("sindrome_metabolico") == 1)
    .group_by("cau_cie10_std") # Agrupamos por el código
    .len()                     # Contamos (en versiones viejas usa .count())
    .sort("len", descending=True) # Ordenamos de mayor a menor
)



In [ ]:
display(resumen_cie10) 

In [ ]:
# Mapeo de Área: 1.0 = URBANO, 2.0 = RURAL
base_general_filtrada = base_general_filtrada.with_columns(
    pl.col("area_res")
    .replace({"1.0": "Urbana", "2.0": "Rural", "1": "Urbana", "2": "Rural"}),

)


In [ ]:
base_general_filtrada["area_res"].value_counts()

# Diferentes outputs

## Base total


In [ ]:
# 2. Define the output file path
output_path = "../../data/data_hospitales/sistema_salud_egresos_limpio.parquet"

# 3. Write the Polars DataFrame to a Parquet file
base_general_filtrada.write_parquet(output_path)

## Resumen columnas res y ubi

In [ ]:
import polars as pl

# Definir las columnas dinámicamente
cols_ubi = [col for col in base_general_filtrada.columns if col.endswith("_ubi")]
cols_res = [col for col in base_general_filtrada.columns if col.endswith("_res")]

# --- Reporte de Ubicación ---
# Agrupamos por todas las columnas de ubicación y contamos
reporte_ubi = (
    base_general_filtrada
    .group_by(cols_ubi)
    .agg(pl.len().alias("num_registros"))
    .sort("num_registros", descending=True)
)

# --- Reporte de Residencia ---
reporte_res = (
    base_general_filtrada
    .group_by(cols_res)
    .agg(pl.len().alias("num_registros"))
    .sort("num_registros", descending=True)
)

reporte_ubi.write_csv("../../data/revision/reporte_ubi.csv")

reporte_res.write_csv("../../data/revision/reporte_res.csv")


In [ ]:

# Mostrar los primeros resultados para verificar
print("Top 5 Combinaciones de Ubicación:")
print(reporte_ubi.head(5))

print("\nTop 5 Combinaciones de Residencia:")
print(reporte_res.head(5))

## Base agregadas

In [ ]:
import pandas as pd
import polars as pl
# 2. Define the output file path
complete_path = "../../data/data_hospitales/sistema_salud_egresos_limpio.parquet"

# 3. Write the Polars DataFrame to a Parquet file
base_general_filtrada= pl.read_parquet(complete_path)


In [ ]:
#value_counts para revisar los valores únicos en las columnas de ubicación y residencia
print("Valores únicos en columnas de ubicación:")
for col in [col for col in base_general_filtrada.columns if col.endswith("_ubi")]:
    print(f"{col}: {base_general_filtrada[col].n_unique()}")

In [ ]:
# cuantos hay de cada provincia
print("\nConteo por provincia de ubicación:")
value_counts=base_general_filtrada["prov_res"].value_counts()

In [ ]:
# filtrar para tener solo los registros con prov_res= "GALAPAGOS" o prov_ubi= "GALAPAGOS"
base_galapagos = base_general_filtrada.filter((pl.col("prov_res") == "GALAPAGOS") | (pl.col("prov_ubi") == "GALAPAGOS"))

In [ ]:
base_galapagos.write_parquet("../../data/revision/base_galapagos.parquet")

In [ ]:
# 1. Aseguramos que las listas de columnas estén presentes para evitar errores
# (Ya las definiste arriba, pero las incluimos en la lógica del group_by)

# 2. Agrupamiento y Conteo
base_general_filtrada_for_group=base_general_filtrada.clone()
DATE_FORMAT = "%Y-%m-%d"
#2. Agrupamiento con conversión de fecha integrada
resumen_egresos = (
    base_general_filtrada_for_group
    .with_columns([
        # Convertimos texto a fecha antes de extraer año/mes
        pl.col("fecha_egr").str.to_date(DATE_FORMAT)
    ])
    .with_columns([
        pl.col("fecha_egr").dt.year().alias("anio_egr"),
        pl.col("fecha_egr").dt.month().alias("mes_egr")
    ])
    .group_by([
        "anio_egr", 
        "mes_egr",
        *identificadores,    # ['clase', 'tipo', 'entidad', 'sector']
        *territorio_ubi,     # ['prov_ubi', 'cant_ubi', 'parr_ubi']
        "cau221rx_std",
        "con_egrpa",
        "edad_std"          # Diagnóstico final
    ])
    .agg([
        pl.len().alias("total_egresos"),
    ])

)

# 3. Verificación de los resultados
print("Estructura del resumen agrupado:")
print(resumen_egresos.head())

In [ ]:
resumen_egresos.write_csv("../../data/processed/resumen_egresos_v3.csv")